# Ejercicio 1
- Cargar el conjunto de datos GLUE con la configuración MRPC e inspeccionar los primeros 5 ejemplos

In [1]:

from datasets import load_dataset

# 1. Cargar la tarea de paráfrasis (equivalente moderno a glue/mrpc)
dataset = load_dataset("google-research-datasets/paws", "labeled_final")

# 2. Inspeccionar los primeros 5 ejemplos del split 'train'
label_names = dataset["train"].features["label"].names

for i in range(5):
    ejemplo = dataset["train"][i]          # <- acceso por índice: devuelve dict
    print(f"\n--- Ejemplo {i+1} ---")
    print(f"id        : {ejemplo['id']}")
    print(f"sentence1 : {ejemplo['sentence1']}")
    print(f"sentence2 : {ejemplo['sentence2']}")
    print(f"label     : {ejemplo['label']} ({label_names[ejemplo['label']]})")


--- Ejemplo 1 ---
id        : 1
sentence1 : In Paris , in October 1560 , he secretly met the English ambassador , Nicolas Throckmorton , asking him for a passport to return to England through Scotland .
sentence2 : In October 1560 , he secretly met with the English ambassador , Nicolas Throckmorton , in Paris , and asked him for a passport to return to Scotland through England .
label     : 0 (0)

--- Ejemplo 2 ---
id        : 2
sentence1 : The NBA season of 1975 -- 76 was the 30th season of the National Basketball Association .
sentence2 : The 1975 -- 76 season of the National Basketball Association was the 30th season of the NBA .
label     : 1 (1)

--- Ejemplo 3 ---
id        : 3
sentence1 : There are also specific discussions , public profile debates and project discussions .
sentence2 : There are also public discussions , profile specific discussions , and project discussions .
label     : 0 (0)

--- Ejemplo 4 ---
id        : 4
sentence1 : When comparable rates of flow can be mai

# Ejercicio 2
- Transmitir (stream) el conjunto de datos C4 y contar cuántos ejemplos puedes procesar en 10 segundos

In [1]:

import time
from datasets import load_dataset

ds = load_dataset("allenai/c4", "en", streaming=True, split="train")
iterador = iter(ds)

start = time.time()
count = 0

for ejemplo in iterador:
    # "procesar" = leer el texto (aquí solo accedemos a la clave 'text')
    _ = ejemplo["text"]
    count += 1
    if time.time() - start >= 10:
        break

print(f"Procesé {count} ejemplos en 10 segundos")

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Procesé 4436 ejemplos en 10 segundos


# Ejercicio 3
- Convertir un conjunto de datos a Parquet y comparar el tamaño del archivo con CSV

In [1]:

from datasets import load_dataset
import os

# 1. Cargar dataset
ds = load_dataset("google-research-datasets/paws", "labeled_final", split="train")

# 2. Guardar en ambos formatos
ds.to_csv("paws.csv")
ds.to_parquet("paws.parquet")

# 3. Comparar tamaños
csv_size = os.path.getsize("paws.csv")
parquet_size = os.path.getsize("paws.parquet")

print(f"CSV:      {csv_size:,} bytes ({csv_size / 1024 / 1024:.1f} MB)")
print(f"Parquet:  {parquet_size:,} bytes ({parquet_size / 1024 / 1024:.1f} MB)")
print(f"Ratio:    CSV/Parquet = {csv_size / parquet_size:.1f}x")

Creating CSV from Arrow format:   0%|          | 0/50 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

CSV:      11,862,475 bytes (11.3 MB)
Parquet:  8,014,796 bytes (7.6 MB)
Ratio:    CSV/Parquet = 1.5x


# Ejercicio 4
- Crear una división de entrenamiento/validación/prueba 70/15/15 con una semilla fija y verificar los tamaños

In [1]:

from datasets import load_dataset

SEED = 42  # semilla global

# 1. Cargar dataset completo
ds = load_dataset("google-research-datasets/paws", "labeled_final", split="train")
total = len(ds)
print(f"Total: {total}")

# 2. Crear splits 70/15/15 reutilizando la misma seed
splits = ds.train_test_split(test_size=0.30, seed=SEED)
final = splits["test"].train_test_split(test_size=0.50, seed=SEED)

train_ds = splits["train"]
val_ds = final["train"]
test_ds = final["test"]

# 3. Verificar tamaños
print(f"Train: {len(train_ds)} ({len(train_ds)/total*100:.1f}%)")
print(f"Val:   {len(val_ds)} ({len(val_ds)/total*100:.1f}%)")
print(f"Test:  {len(test_ds)} ({len(test_ds)/total*100:.1f}%)")

Total: 49401
Train: 34580 (70.0%)
Val:   7410 (15.0%)
Test:  7411 (15.0%)
